# Baseline — Cenário 2: CSV + PostgreSQL

**Spec equivalente:**  
> Leia `orders.csv` e a tabela `public.customers` do PostgreSQL. Faça join em `customer_id`. Mantenha apenas `status == 'active'`. Renomeie `order_dt` → `order_date`. Remova linhas com `customer_id` nulo. Salve em `public.orders_cleaned`.

---

## Métricas de implementação

| Métrica | Valor |
|---|---|
| Tempo de implementação | ~10 min |
| Linhas de código | ver célula final |
| Decisões explícitas | 5 (tipo de join, coluna de join, tipo de rename, qual coluna é nula, tabela destino) |
| Erros durante desenvolvimento | 1 (SQLAlchemy connection string) |
| Iterações | 2 |

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(dotenv_path=Path("../../.env"))
POSTGRES_URL = os.getenv("POSTGRES_URL")
engine = create_engine(POSTGRES_URL)

start = time.time()

### 1. Extração

In [2]:
orders = pd.read_csv("../data/orders.csv")
print(f"orders: {orders.shape}")

with engine.connect() as conn:
    customers = pd.read_sql(text("SELECT * FROM public.customers"), conn)
print(f"customers: {customers.shape}")

orders: (10000, 7)


customers: (400, 4)


### 2. Transformação

In [3]:
# Remove null customer_id before join
orders = orders[orders["customer_id"].notnull()]
orders["customer_id"] = orders["customer_id"].astype(int)

# Join
df = orders.merge(customers, on="customer_id", how="inner")

# Filter active
df = df[df["status"] == "active"]

# Rename
df = df.rename(columns={"order_dt": "order_date"})

df = df.reset_index(drop=True)
print(f"After transform: {df.shape}")
df.head(3)

After transform: (7135, 10)


,order_id,customer_id,order_date,product,amount,qty,status,name,city,email
0,11253,210,2024-09-17,Device Pro,785.21,15,active,Bruno Lima,Recife,customer210@example.com
1,9685,306,2024-07-14,Gadget Y,408.17,19,active,Carlos Santos,Olinda,customer306@example.com
2,6732,461,2024-03-13,Gadget X,54.00,21,active,Igor Santos,Recife,customer461@example.com


### 3. Qualidade (manual)

In [4]:
null_report = df.isnull().mean().round(4)
print("Null ratios (non-zero):")
print(null_report[null_report > 0])
print(f"\nDuplicates: {df.duplicated().sum()}")

Null ratios (non-zero):
Series([], dtype: float64)

Duplicates: 134


### 4. Carga

In [5]:
df.to_sql("orders_cleaned", engine, schema="public", if_exists="replace", index=False)

elapsed = time.time() - start
print(f"Saved {len(df)} rows → public.orders_cleaned")
print(f"Pipeline execution time: {elapsed:.2f}s")

Saved 7135 rows → public.orders_cleaned
Pipeline execution time: 1.21s


---
## Resumo de métricas


In [6]:
code_lines = 16

print("=" * 50)
print("BASELINE — Cenário 2")
print("=" * 50)
print(f"Rows output:              {len(df)}")
print(f"Execution time:           {elapsed:.2f}s")
print(f"LOC written:              {code_lines}")
print(f"Explicit decisions:       5")
print(f"Dev errors (tracebacks):  1")
print(f"Iterations to correct:    2")
print()
print("AI-ETL (avg of 5 runs):")
print(f"  Execution time:         16.1s")
print(f"  LOC written by human:   0 (spec in NL)")
print(f"  LLM attempts:           2.0")
print(f"  Human interventions:    0")

BASELINE — Cenário 2
Rows output:              7135
Execution time:           1.21s
LOC written:              16
Explicit decisions:       5
Dev errors (tracebacks):  1
Iterations to correct:    2

AI-ETL (avg of 5 runs):
  Execution time:         16.1s
  LOC written by human:   0 (spec in NL)
  LLM attempts:           2.0
  Human interventions:    0
